# Evaluation: FID, SSIM, PSNR, LPIPS

Computes the image-quality metrics from chapter 6.4.2 for one generation run
and writes the results as CSV (one aggregate row + an optional per-sample file
for distribution plots in LaTeX).

Three modes, set via the `MODE` knob below:

- `unconditional`       — random real↔gen pairs, all four metrics
- `seg2mri`             — exact pairs via `stats.json` cond stems, all four metrics
- `novel_segmentation`  — FID only (no meaningful reference for SSIM/PSNR/LPIPS)

Re-run the notebook once per (MODE, MODEL_NAME, SAMPLER) combination. Each
run produces its own CSV in `results/`.

In [ ]:
from pathlib import Path

MODE       = "unconditional"   # "unconditional" | "seg2mri" | "novel_segmentation"
MODEL_NAME = "baseline"        # any tag — shows up in the CSV and filename
SAMPLER    = "DDIM"
IMAGE_SIZE = 32

# Real reference volumes. For seg2mri this is the T1N directory; for
# novel_segmentation it is the seg-mask directory.
REAL_DIR = Path(rf"C:\Users\Sven\Desktop\Data\Processed\brats\{IMAGE_SIZE}")

# Generated samples from generate.ipynb / generate_conditional.ipynb.
GEN_DIR = Path(rf"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\unconditional\samples\{IMAGE_SIZE}\{SAMPLER}\samples_{MODEL_NAME}\samples")

# stats.json lives next to the samples/ folder — only used in seg2mri mode.
STATS_JSON = GEN_DIR.parent / "stats.json"

RESULTS_DIR = Path.cwd() / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"mode={MODE}  model={MODEL_NAME}  sampler={SAMPLER}  size={IMAGE_SIZE}")
print(f"real: {REAL_DIR}")
print(f"gen:  {GEN_DIR}")

In [ ]:
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))

import volumes as vol
import metrics as mtr

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

In [ ]:
real_files = vol.list_volumes(REAL_DIR)
gen_files  = vol.list_volumes(GEN_DIR)
print(f"found {len(real_files)} real, {len(gen_files)} generated")

if MODE == "seg2mri":
    pairs = vol.pair_by_stats(REAL_DIR, gen_files, STATS_JSON)
else:
    pairs = vol.pair_random(real_files, gen_files, seed=42)

print(f"prepared {len(pairs)} pairs")

In [ ]:
# Load every volume that's needed into memory once. For 1024 paired volumes
# at 32³ this is ~135 MB — well within limits and saves re-reading from disk
# between SSIM/PSNR/LPIPS and FID.
real_paired = [vol.load_volume(r) for r, _ in tqdm(pairs, desc="load real (paired)")]
gen_paired  = [vol.load_volume(g) for _, g in tqdm(pairs, desc="load gen")]

# FID benefits from the full real set, not just the paired subsample. Load
# the rest only if there are more reals than we already have.
if MODE == "seg2mri" or len(real_files) <= len(pairs):
    real_all = real_paired
else:
    real_all = [vol.load_volume(f) for f in tqdm(real_files, desc="load real (full)")]

print(f"in memory: {len(real_paired)} paired real, {len(gen_paired)} gen, {len(real_all)} reals for FID")

In [ ]:
# Per-sample SSIM/PSNR/LPIPS. Skipped for novel_segmentation since there is
# no meaningful ground-truth pair (the generated seg-map is conditioned only
# on a size class).
per_sample_rows = []

if MODE != "novel_segmentation":
    lpips_model = mtr.load_lpips(device)

    for i, (real, gen) in enumerate(tqdm(list(zip(real_paired, gen_paired)), desc="ssim/psnr/lpips")):
        per_sample_rows.append({
            "sample_id": i + 1,
            "ssim":  mtr.ssim(real, gen),
            "psnr":  mtr.psnr(real, gen),
            "lpips": mtr.lpips_score(real, gen, lpips_model, device),
        })

    del lpips_model
    if device == "cuda":
        torch.cuda.empty_cache()

    per_sample_df = pd.DataFrame(per_sample_rows)
    print(per_sample_df.describe())
else:
    per_sample_df = pd.DataFrame()
    print("skipped — novel_segmentation mode (FID only)")

In [ ]:
inception = mtr.load_inception(device)
fid_score = mtr.fid(real_all, gen_paired, inception, device)

del inception
if device == "cuda":
    torch.cuda.empty_cache()

print(f"FID = {fid_score:.4f}  (over {len(real_all)} real vs {len(gen_paired)} generated)")

In [ ]:
tag = f"{MODE}_{MODEL_NAME}_{SAMPLER}_{IMAGE_SIZE}"

# Per-sample CSV — only written when there are pairwise rows. Useful for
# boxplots / distribution figures in LaTeX (\pgfplotstable).
if not per_sample_df.empty:
    per_sample_path = RESULTS_DIR / f"per_sample_{tag}.csv"
    per_sample_df.to_csv(per_sample_path, index=False, float_format="%.6f")
    print(f"wrote {per_sample_path}")

# Aggregate row — one CSV per run, easy to concatenate later in LaTeX.
agg = {
    "mode":       MODE,
    "model":      MODEL_NAME,
    "sampler":    SAMPLER,
    "image_size": IMAGE_SIZE,
    "n_pairs":    len(pairs),
    "n_real_fid": len(real_all),
    "fid":        fid_score,
}
if not per_sample_df.empty:
    for col in ("ssim", "psnr", "lpips"):
        agg[f"{col}_mean"] = float(per_sample_df[col].mean())
        agg[f"{col}_std"]  = float(per_sample_df[col].std())

agg_path = RESULTS_DIR / f"metrics_{tag}.csv"
pd.DataFrame([agg]).to_csv(agg_path, index=False, float_format="%.6f")
print(f"wrote {agg_path}")

In [ ]:
# Quick sanity check — not for the thesis, just to see at a glance whether
# the run looks reasonable before moving on to the next model.
print(f"\n=== {tag} ===")
for k, v in agg.items():
    if isinstance(v, float):
        print(f"  {k:<14} {v:.4f}")
    else:
        print(f"  {k:<14} {v}")